# **Part 1 -  INGESTION ENGINE**

### Install Dependencies

In [ ]:
!pip install -q \
    qdrant-client \
    sentence-transformers \
    transformers \
    tiktoken \
    ragas \
    datasets \
    jedi>=0.16 \
    "protobuf<6.0.0" \
    "requests>=2.32.4"

### Imports

In [ ]:
import os
import re
import uuid
import tiktoken

from typing import List, Dict

from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct, Filter, FieldCondition, MatchValue

from sentence_transformers import SentenceTransformer, CrossEncoder

### Initialize Models

In [ ]:
embed_model = SentenceTransformer("BAAI/bge-large-en-v1.5")
reranker = CrossEncoder("BAAI/bge-reranker-large")

encoding = tiktoken.get_encoding("cl100k_base")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


### Initialize Vector DB(Qdrant)

In [ ]:
qdrant = QdrantClient(":memory:")

COLLECTION = "aegis_rag"

qdrant.recreate_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE)
)

/tmp/ipykernel_39306/3850115622.py:5: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(


True

### Token Counter

In [ ]:
def count_tokens(text):
    return len(encoding.encode(text))

### Markdown Header Split

In [ ]:
def split_markdown_headers(text):
    pattern = r"(#+ .+)"
    parts = re.split(pattern, text)

    chunks = []
    h1, h2 = "ROOT", "ROOT"

    for part in parts:
        if part.startswith("## "):
            h2 = part.strip()
        elif part.startswith("# "):
            h1 = part.strip()
            h2 = "ROOT"
        else:
            if part.strip():
                chunks.append({
                    "h1": h1,
                    "h2": h2,
                    "content": part.strip()
                })

    return chunks

### Table Detection

In [ ]:
def is_table_block(text):
    return "|" in text and "---" in text

### Token based chunking with Overlap

In [ ]:
def chunk_with_overlap(text, max_tokens=300, overlap=50):
    tokens = encoding.encode(text)
    chunks = []

    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]

        chunk_text = encoding.decode(chunk_tokens)
        chunks.append(chunk_text)

        start += (max_tokens - overlap)

    return chunks

### Metadata Extraction

In [ ]:
def extract_date(text):
    match = re.search(r"Effective Date:\s*(.+)", text)
    return match.group(1) if match else "1900-01-01"


def infer_category(filename):
    f = filename.lower()

    if "travel" in f:
        return "Travel"
    elif "security" in f:
        return "Security"
    elif "leave" in f or "performance" in f:
        return "HR"
    else:
        return "General"

### Ingestion Pipeline

In [ ]:
def ingest(text, filename):
    sections = split_markdown_headers(text)
    points = []

    for sec in sections:
        content = sec["content"]

        # Table-safe logic
        if is_table_block(content):
            chunks = [content]
        else:
            chunks = chunk_with_overlap(content)

        for chunk in chunks:
            vector = embed_model.encode(chunk).tolist()

            payload = {
                "text": chunk,
                "document_id": filename,
                "policy_category": infer_category(filename),
                "effective_date": extract_date(text),
                "h1": sec["h1"],
                "h2": sec["h2"]
            }

            points.append(PointStruct(
                id=str(uuid.uuid4()),
                vector=vector,
                payload=payload
            ))

    qdrant.upsert(collection_name=COLLECTION, points=points)

### Upload Files

In [ ]:
from google.colab import files
files.upload()

Saving it security and data privacy.txt to it security and data privacy (1).txt


{'it security and data privacy (1).txt': b'# Global IT Security, Data Classification, and Acceptable Use Policy\r\n**Document ID:** SEC-POL-8005-V7\r\n**Effective Date:** June 1, 2026\r\n**Last Revised:** April 15, 2026\r\n**Policy Owner:** Chief Information Security Officer (CISO)\r\n**Applies To:** All employees, contractors, third-party vendors, and any entity with provisioned access to the corporate network or data infrastructure.\r\n\r\n---\r\n\r\n## Table of Contents\r\n1. Security Philosophy and Zero-Trust Architecture\r\n2. Data Classification Matrix\r\n3. Acceptable Use of Corporate Systems\r\n4. Bring Your Own Device (BYOD) and Mobile Management\r\n5. Password Policies and Authentication\r\n6. Data Privacy and Handling of PII/Financial Data\r\n7. Incident Response and Breach Reporting SLAs\r\n8. Software Procurement and "Shadow IT"\r\n9. Offboarding and Access Revocation\r\n\r\n---\r\n\r\n## 1. Security Philosophy and Zero-Trust Architecture\r\nThe organization operates on a 

In [ ]:
from google.colab import files
files.upload()

Saving learning and tuition.txt to learning and tuition (1).txt


{'learning and tuition (1).txt': b'# Corporate Policy: Global Learning, Development, and Tuition Assistance\r\n**Document ID:** LND-POL-7010-V3\r\n**Effective Date:** May 1, 2026\r\n**Last Revised:** March 10, 2026\r\n**Policy Owner:** Chief Learning Officer (CLO) & Global HR\r\n**Applies To:** All full-time employees. Part-time employees and contractors are subject only to Section 2 (Mandatory Compliance).\r\n\r\n---\r\n\r\n## Table of Contents\r\n1. Learning Philosophy and Scope\r\n2. Mandatory Compliance and Onboarding Training\r\n3. The Professional Development Stipend (Certifications & Conferences)\r\n4. Advanced Technology & AI Fluency Initiative\r\n5. Formal Tuition Assistance Program (Degree Programs)\r\n6. Academic Performance Requirements\r\n7. Reimbursement Workflows\r\n8. Repayment and Retention (The "Clawback" Clause)\r\n9. Cross-References and Taxation\r\n\r\n---\r\n\r\n## 1. Learning Philosophy and Scope\r\nThe organization is committed to fostering a culture of continuo

In [ ]:
from google.colab import files
files.upload()

Saving code of conduct.txt to code of conduct (1).txt
Saving leave_and_absence.txt to leave_and_absence (1).txt
Saving performance and compensation.txt to performance and compensation (1).txt


{'code of conduct (1).txt': b'# Human Resources Policy: Global Code of Conduct and Disciplinary Procedures\r\n**Document ID:** HR-POL-5050-V4\r\n**Effective Date:** March 1, 2026\r\n**Last Revised:** January 15, 2026\r\n**Policy Owner:** Chief Human Resources Officer (CHRO) & Office of the General Counsel\r\n**Applies To:** All global employees, board members, independent contractors, and vendors operating on company premises or accessing company networks.\r\n\r\n---\r\n\r\n## Table of Contents\r\n1. Core Philosophy and Expected Standards\r\n2. Reporting Mechanisms and The Ethics Hotline\r\n3. Non-Retaliation and Whistleblower Protection\r\n4. Investigation Protocols and Timelines\r\n5. Categorization of Policy Violations\r\n6. Progressive Disciplinary Process\r\n7. Zero-Tolerance Behaviors and Immediate Termination\r\n8. Anti-Harassment and Non-Discrimination Policy\r\n9. Conflicts of Interest, Gifts, and Bribery\r\n10. Substance Abuse and Alcohol Consumption\r\n11. Appeals and Grieva

In [ ]:
from google.colab import files
files.upload()

Saving Fuel and Mileage Policy.txt to Fuel and Mileage Policy (1).txt
Saving International Travel.txt to International Travel (1).txt
Saving Travel Policy.txt to Travel Policy (1).txt


{'Fuel and Mileage Policy (1).txt': b'# Corporate Travel Policy: Personal Vehicle, Fuel, and Mileage Reimbursement\r\n**Document ID:** TRV-POL-3012-V2\r\n**Effective Date:** April 1, 2026\r\n**Last Revised:** January 20, 2026\r\n**Policy Owner:** Corporate Fleet Management & Global Finance\r\n**Applies To:** All employees authorized to operate personal, rental, or company-owned vehicles for official business purposes.\r\n\r\n---\r\n\r\n## Table of Contents\r\n1. Purpose and Guiding Principles\r\n2. Definitions and Scope\r\n3. Personal Vehicle Usage: Standard Mileage Rate (SMR) Program\r\n4. Personal Vehicle Usage: Fixed and Variable Rate (FAVR) Allowance\r\n5. Fuel Reimbursement for Rental and Company-Owned Vehicles\r\n6. Electric Vehicle (EV) and Hybrid Fleet Guidelines\r\n7. Chauffeur and Professional Driver Allowances\r\n8. The "Normal Commute" Deduction Rule\r\n9. Tolls, Parking, and Ancillary Transit Costs\r\n10. Documentation, Telematics, and Odometer Verification\r\n11. Insuranc

### Load & Ingest

In [ ]:
for file in os.listdir("/content"):
    if file.endswith(".txt"):
        with open(f"/content/{file}", "r", encoding="utf-8", errors="ignore") as f:
            ingest(f.read(), file)
            print("Ingested:", file)

Ingested: leave_and_absence (1).txt
Ingested: learning and tuition.txt
Ingested: Travel Policy.txt
Ingested: Fuel and Mileage Policy (1).txt
Ingested: International Travel (1).txt
Ingested: Travel Policy (1).txt
Ingested: performance and compensation.txt
Ingested: code of conduct (1).txt
Ingested: learning and tuition (1).txt
Ingested: leave_and_absence.txt
Ingested: performance and compensation (1).txt
Ingested: it security and data privacy.txt
Ingested: International Travel.txt
Ingested: Fuel and Mileage Policy.txt
Ingested: code of conduct.txt
Ingested: it security and data privacy (1).txt


# **PART 2 — ADVANCED RETRIEVAL PIPELINE**

### Query Expansion + HyDE

In [ ]:
def expand_query(q):
    return [
        q,
        f"Policy regarding {q}",
        f"Corporate rules for {q}",
        f"Guidelines about {q}",
        f"This document explains {q}"
    ]

### Intent Detection (Pre-Filter)

In [ ]:
def detect_category(q):
    q = q.lower()

    if "leave" in q:
        return "HR"
    elif "travel" in q or "taxi" in q:
        return "Travel"
    elif "password" in q or "security" in q:
        return "Security"

    return None

### Retrieval

In [ ]:
def retrieve(query):
    queries = expand_query(query)
    category = detect_category(query)

    all_hits = []

    for q in queries:
        vec = embed_model.encode(q).tolist()

        filt = None
        if category:
            filt = Filter(
                must=[FieldCondition(key="policy_category", match=MatchValue(value=category))]
            )

        hits = qdrant.search(
            collection_name=COLLECTION,
            query_vector=vec,
            limit=10,
            query_filter=filt
        )

        all_hits.extend(hits)

    return deduplicate(all_hits)

### Deduplication + Post Filtering

In [ ]:
def deduplicate(hits):
    seen = set()
    unique = []

    for h in hits:
        txt = h.payload["text"]
        if txt not in seen:
            seen.add(txt)
            unique.append(h)

    return unique


def post_filter(chunks):
    latest = {}

    for c in chunks:
        doc = c.payload["document_id"]
        date = c.payload["effective_date"]

        if doc not in latest or date > latest[doc][1]:
            latest[doc] = (c, date)

    return [v[0] for v in latest.values()]

### Reranking(Cross-Encoder)

In [ ]:
def rerank(query, chunks):
    if not chunks:
        return []

    pairs = [(query, c.payload["text"]) for c in chunks]
    scores = reranker.predict(pairs)

    ranked = sorted(zip(chunks, scores), key=lambda x: x[1], reverse=True)

    return [r[0] for r in ranked[:5]]

### Guardrails

In [ ]:
def is_malicious(q):
    return any(x in q.lower() for x in ["ignore previous", "bypass", "override"])

### Answer Generation(Grounded)

In [ ]:
def generate_answer(query, chunks):
    if not chunks:
        return "No relevant policy found."

    context = "\n\n".join([c.payload["text"] for c in chunks])

    return f"""
STRICT POLICY ANSWER:

Question: {query}

Answer:
{context[:1200]}

NOTE:
- Answer is strictly grounded in retrieved policy.
- No hallucinated content included.
"""

### Full Pipeline

In [ ]:
def rag_pipeline(query):
    if is_malicious(query):
        return "Query blocked due to security policy.", []

    chunks = retrieve(query)
    chunks = post_filter(chunks)
    chunks = rerank(query, chunks)

    answer = generate_answer(query, chunks)

    return answer, chunks

# **PART 3 — STREAMLIT APP**

### Streamlit App

In [ ]:
!pip install -q streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 41.4 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
import streamlit as st

st.title("Enterprise RAG - Project Aegis")

query = st.text_input("Ask your policy question:")

if query:
    st.write("Your RAG pipeline output will appear here")

Writing app.py


### Add ngrok token

In [ ]:
# Set your ngrok auth token here locally; do not commit secrets.
# ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN")

### Run Streamlit in background

In [ ]:
!streamlit run app.py &>/dev/null &

### Expose via ngrok

In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print("Open this URL:", public_url)

Open this URL: NgrokTunnel: "https://ruby-charting-starry.ngrok-free.dev" -> "http://localhost:8501"
